# Contact Session 5 — Introduction to Django

**Topics:** What is Django, Project setup, App vs Project, Models, Views, URLs, Settings, Migrations, Templates

**Assignment:** Build a multimedia website Django project (take-home)

---

## 1. What is Django?

Django is a Python web framework. A framework is just a set of tools someone already built so you don't have to write everything from scratch.

Django is called **batteries-included** because it comes with:
- A URL router
- A database ORM (talk to your DB with Python, no raw SQL needed)
- A templating engine
- An auto-generated admin panel
- Authentication
- Security features baked in

You're not piecing together ten different libraries. It's all there.

### The pattern Django follows: MVT

| Layer | File | Job |
|-------|------|-----|
| Model | `models.py` | Defines your database tables |
| View | `views.py` | Handles the logic for each page |
| Template | `.html` files | What the user actually sees |

### How a request flows through Django:

```
Browser sends request
       ↓
   urls.py  → matches the URL to a view
       ↓
   views.py → runs the logic, asks models for data
       ↓
   models.py → fetches from the database
       ↓
   template → renders HTML with the data
       ↓
Browser receives the page
```

---

## 2. Project vs App

This confuses everyone at first. Here's the simple version:

| | Project | App |
|-|---------|-----|
| What it is | The whole website | One feature of the website |
| Command | `django-admin startproject` | `python manage.py startapp` |
| Example | `mysite` | `blog`, `accounts`, `shop` |

A project contains one or more apps. An app is meant to do one thing well.



---

## 3. Setup — Installing Django

In [ ]:
!pip install django

In [ ]:
# Confirm it installed correctly
!python -m django --version

## 4. Create the Project

In [ ]:
!django-admin startproject mysite

### What are these files?

```
mysite/
├── manage.py          ← your command-line tool for everything Django
└── mysite/
    ├── settings.py    ← config for the whole project
    ├── urls.py        ← root URL dispatcher
    ├── wsgi.py        ← don't touch (for deployment)
    └── asgi.py        ← don't touch (for async deployment)
```

You will spend most of your time in `settings.py` and `urls.py` at the project level.

---

## 5. Create the App

In [ ]:

!python manage.py startapp blog

### What are these files?

```
blog/
├── models.py      ← define your database tables here
├── views.py       ← write your page logic here
├── admin.py       ← register models with the admin panel
├── apps.py        ← app config (don't touch)
├── tests.py       ← write tests here (not today)
└── migrations/    ← auto-generated DB migration files
```

Notice there is no `urls.py` inside the app. You create that yourself.

---

## 6. settings.py

`settings.py` is the control center for your project. Here are the parts you care about right now:

### `INSTALLED_APPS`
Any app you create must be registered here or Django ignores it completely.

```python
INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
    'blog',   # <-- add your app here
]
```

### `DATABASES`
By default Django uses SQLite. Fine for development.

```python
DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.sqlite3',
        'NAME': BASE_DIR / 'db.sqlite3',
    }
}
```

### `DEBUG`
`True` in development (gives you detailed error pages). `False` in production, always.

### `TEMPLATES`
Tells Django where to look for your HTML files. As long as `APP_DIRS` is `True`, Django will automatically find templates inside each app's `templates/` folder.

### The rule to remember
Any time you create something new — a new app, a new template folder, a static files folder — check if `settings.py` needs to know about it. It usually does.

---

## 7. Register the App in settings.py

In [ ]:
INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
    'blog',
]


## 8. Models

A model is a Python class that maps to a database table. Each attribute on the class is a column in the table.

You don't write SQL. Django handles that through the ORM.

Common field types:

| Field | What it stores |
|-------|----------------|
| `CharField(max_length=n)` | Short text |
| `TextField()` | Long text |
| `IntegerField()` | Whole numbers |
| `DateTimeField()` | Date and time |
| `BooleanField()` | True / False |
| `ForeignKey()` | Link to another model |

In [ ]:
from django.db import models

class Post(models.Model):
    title = models.CharField(max_length=200)
    content = models.TextField()
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return self.title

## 9. Migrations

You defined the model in Python, but the database doesn't know about it yet.

Migrations are how Django keeps your database in sync with your models.

Two commands, always in this order:

1. `makemigrations` — detects changes in your models and creates a migration file
2. `migrate` — reads those migration files and applies them to the actual database

Think of `makemigrations` as writing the instructions, and `migrate` as carrying them out.

In [ ]:
!python manage.py makemigrations

In [ ]:
!python manage.py migrate

Your database now has a `blog_post` table with the columns you defined.

---

## 10. Views

A view is a Python function that receives a request and returns a response.

That's it. Every page on your site is a view.

The pattern is always:
```python
def my_view(request):
    # do something
    return render(request, 'template.html', context)
```

`context` is just a dictionary of data you want to pass to the template.

In [ ]:
from django.shortcuts import render
from .models import Post

def post_list(request):
    posts = Post.objects.all().order_by('-created_at')
    return render(request, 'blog/post_list.html', {'posts': posts})

## 11. URLs

Django needs to know which URL should trigger which view.

There are two levels of URLs:
- The **project-level** `urls.py` is the root dispatcher. It forwards traffic to each app.
- The **app-level** `urls.py` handles the routes specific to that app. You create this file yourself.

```
Request: /blog/
  → mysite/urls.py says: anything starting with '' → go to blog/urls.py
  → blog/urls.py says: '' → call post_list view
```

In [ ]:
from django.urls import path
from . import views

urlpatterns = [
    path('', views.post_list, name='post_list'),
]


In [ ]:
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('blog.urls')),
]

## 12. Templates

Templates are HTML files with Django's templating language mixed in.

### The syntax you need to know:

| Syntax | What it does |
|--------|--------------|
| `{{ variable }}` | Print a variable |
| `{% for x in list %}` | Loop |
| `{% endfor %}` | End the loop |
| `{% if condition %}` | Conditional |
| `{% endif %}` | End the conditional |
| `{% empty %}` | Runs if a for loop has nothing to iterate |
| `{% extends 'base.html' %}` | Inherit from another template |
| `{% block content %}` | Define a replaceable section |

Django looks for templates inside `appname/templates/appname/`. The double folder is a Django convention to avoid name conflicts between apps.

# Create the templates directory


In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>My Blog</title>
    <style>
        body {
            font-family: sans-serif;
            max-width: 680px;
            margin: 60px auto;
            padding: 0 20px;
            color: #222;
        }
        h1 {
            font-size: 2rem;
            margin-bottom: 40px;
        }
        .post {
            border-bottom: 1px solid #eee;
            padding: 24px 0;
        }
        .post h2 {
            margin: 0 0 10px;
            font-size: 1.3rem;
        }
        .post p {
            margin: 0 0 10px;
            line-height: 1.6;
            color: #444;
        }
        .post small {
            color: #999;
            font-size: 0.85rem;
        }
        .empty {
            color: #999;
            font-style: italic;
        }
    </style>
</head>
<body>
    <h1>Blog Posts</h1>

    {% for post in posts %}
        <div class="post">
            <h2>{{ post.title }}</h2>
            <p>{{ post.content }}</p>
            <small>Posted on {{ post.created_at }}</small>
        </div>
    {% empty %}
        <p class="empty">No posts yet. Add some from the shell.</p>
    {% endfor %}

</body>
</html>

python manage.py shell

In [ ]:
from blog.models import Post

Post.objects.create(title="First Post", content="Hello Django. This actually works.")
Post.objects.create(title="Second Post", content="Adding a second post to see the loop in action.")
Post.objects.create(title="Third Post", content="Three posts. The template is looping correctly.")

Post.objects.count()

## 14. Run the Server

In [ ]:
python manage.py runserver

## 15. Full File Structure (final state)

```
mysite/
├── manage.py
├── db.sqlite3
├── mysite/
│   ├── settings.py     ← registered 'blog' in INSTALLED_APPS
│   └── urls.py         ← includes blog.urls
│
└── blog/
    ├── models.py       ← Post model
    ├── views.py        ← post_list view
    ├── urls.py         ← path to post_list  (you created this)
    ├── templates/
    │   └── blog/
    │       └── post_list.html
    └── migrations/
        └── 0001_initial.py
```

---

## 16. Quick Reference — Commands

| Command | What it does |
|---------|-------------|
| `django-admin startproject name` | Create a new project |
| `python manage.py startapp name` | Create a new app |
| `python manage.py makemigrations` | Detect model changes, create migration files |
| `python manage.py migrate` | Apply migrations to the database |
| `python manage.py runserver` | Start the development server |
| `python manage.py shell` | Open an interactive Python shell with Django loaded |
| `python manage.py createsuperuser` | Create an admin account |

---

## Assignment (Take-Home)

**Build a multimedia website Django project.**

Requirements:
- At least 2 apps (e.g. `gallery` and `videos`, or `articles` and `music`)
- At least one model per app
- Working templates for each app
- Navigation between pages
- Populate with real data (no dummy placeholder content)

The goal is for you to repeat the same process from this session yourself without following instructions step by step. If you can build this without looking at the notes, you understand Django's setup.

---

*Session 5 complete.*